# Train half-Cu unscaled Figshare pipeline

Trains the scratch M3GNet encoder, exports features, trains UniversalXAS and tuned heads, and evaluates on a material-level half-Cu training subset. All validation and test materials are unchanged; all other elements are unchanged.

The half-Cu split is saved to `output/training/figshare_141_unscaled_halfcu/dataset/halfcu.txt`.

Both processes inherit the GPU from the environment. Set `CUDA_VISIBLE_DEVICES` before starting Jupyter, for example `CUDA_VISIBLE_DEVICES=7 jupyter notebook`. This notebook does not select or inspect GPUs.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def find_repo_root():
    cwd = Path.cwd().resolve()
    if (cwd / 'tutorial_omnixas').is_dir():
        return cwd
    if cwd.name == 'tutorial_omnixas' and (cwd.parent / 'omnixas').is_dir():
        return cwd.parent
    raise FileNotFoundError('Start Jupyter from the OmniXAS repository root or tutorial_omnixas directory.')

REPO_ROOT = find_repo_root()
DATA_ROOT = Path(os.environ.get('OMNIXAS_DATA_ROOT', REPO_ROOT.parent / 'OmniXAS_data')).expanduser()
RAW_ROOT = DATA_ROOT / 'figshare'
NEW_ROOT = DATA_ROOT / 'figshare_141_splits_unscaled'
OUTPUT_ROOT = REPO_ROOT / 'output' / 'training' / 'figshare_141_unscaled_halfcu'
DATASET_DIR = OUTPUT_ROOT / 'dataset'
HALF_SPLIT_PATH = DATASET_DIR / 'halfcu.txt'
ANALYSIS_DIR = REPO_ROOT / 'reports' / 'figshare_unscaled_halfcu_analysis'
ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu']
SPLITS = ['train', 'val', 'test']
RUN_NAME = 'halfcu'
RUN_DIR = OUTPUT_ROOT / RUN_NAME
SCRIPT = REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py'
SEED = 42
HEAD_DROPOUT = 0.05
UNIVERSAL_LOSS_BALANCE_POWER = 1.5
UNIVERSAL_LEARNING_RATE = 0.002
UNIVERSAL_SEED = 43
TUNED_LEARNING_RATE = 0.001
TUNED_SEED = 42

required = [
    REPO_ROOT / 'models' / 'M3GNet-MP-2021.2.8-PES' / 'model.pt',
    REPO_ROOT / 'omnixas' / 'featurizer' / 'm3gnet_featurizer.py',
    SCRIPT,
    RAW_ROOT / 'manifest.csv',
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Required input is missing: {path}')

print(f'Repository: {REPO_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Run directory: {RUN_DIR}')

In [ ]:
def read_ids(path):
    rows = [line.strip() for line in path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
    if not rows:
        raise ValueError(f'Empty ID split: {path}')
    return rows

def read_targets(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing target file: {path}')
    values = np.loadtxt(path, dtype=np.float32, ndmin=2)
    if values.shape[1] != 141 or not np.isfinite(values).all():
        raise ValueError(f'Targets must be finite with 141 columns: {path} {values.shape}')
    if (values < 0).any():
        raise ValueError(f'Targets must be nonnegative: {path}')
    return values

def ensure_unscaled_splits():
    required_splits = [
        NEW_ROOT / folder / f'{element}_FEFF_{split}{suffix}.txt'
        for folder, suffix in (('material_id_and_site', ''), ('ml_data', '_y'))
        for element in ELEMENTS
        for split in SPLITS
    ]
    missing = [path for path in required_splits if not path.is_file()]
    if missing:
        if NEW_ROOT.exists() and any(NEW_ROOT.rglob('*')):
            raise FileNotFoundError(f'Unscaled split tree is incomplete. Move or repair {NEW_ROOT}.')
        exporter = REPO_ROOT / 'tutorial_omnixas' / 'export_figshare_141_splits_unscaled.py'
        subprocess.run([sys.executable, str(exporter)], cwd=REPO_ROOT, check=True)
    missing = [path for path in required_splits if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'Export did not create {missing[0]}')

ensure_unscaled_splits()
report_path = NEW_ROOT / 'report.json'
manifest_path = NEW_ROOT / 'manifest.csv'
if not report_path.is_file() or not manifest_path.is_file():
    raise FileNotFoundError(f'Unscaled exporter report or manifest is missing under {NEW_ROOT}')
report = json.loads(report_path.read_text(encoding='utf-8'))
if report.get('intensity_normalization') != 'none (factor 1.0; no xsedge-derived scale)':
    raise ValueError(f'Unexpected target normalization: {report.get("intensity_normalization")!r}')
row_manifest = pd.read_csv(manifest_path)
if not np.all(pd.to_numeric(row_manifest['target_scale_factor']) == 1.0):
    raise ValueError('The unscaled manifest contains a target scale other than 1.0')

new_data = {}
for element in tqdm(ELEMENTS, desc='Validate elements'):
    material_sets = {}
    for split in SPLITS:
        key = (element, split)
        ids = read_ids(NEW_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt')
        targets = read_targets(NEW_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt')
        if len(ids) != len(targets):
            raise ValueError(f'ID/target mismatch for {key}: {len(ids)} != {len(targets)}')
        rows = row_manifest[(row_manifest['element'] == element) & (row_manifest['split'] == split)].sort_values('row_index')
        if rows['material_id_and_site'].tolist() != ids:
            raise ValueError(f'Manifest/ID ordering mismatch for {key}')
        materials = {value.rsplit('_', 1)[0] for value in ids}
        for value in ids:
            material_id, site = value.rsplit('_', 1)
            poscar = RAW_ROOT / 'FEFF' / element / material_id / 'FEFF-XANES' / f'{int(site):03d}_{element}' / 'POSCAR'
            if not poscar.is_file():
                raise FileNotFoundError(f'No POSCAR for {value}: {poscar}')
        material_sets[split] = materials
        new_data[key] = {'ids': ids, 'y': targets}
    for left_index, split in enumerate(SPLITS):
        for other in SPLITS[left_index + 1:]:
            overlap = material_sets[split] & material_sets[other]
            if overlap:
                raise ValueError(f'Material leakage for {element}: {split}/{other}: {sorted(overlap)[:5]}')
    print(f'{element}: {[len(new_data[(element, split)]["ids"]) for split in SPLITS]}')

if RUN_DIR.exists():
    raise FileExistsError(f'Run directory already exists. Choose a new run name or remove it first: {RUN_DIR}')
if HALF_SPLIT_PATH.exists():
    raise FileExistsError(f'Half-Cu split already exists. Inspect or remove it first: {HALF_SPLIT_PATH}')

In [ ]:
# Select about half of Cu training rows while keeping complete materials together.
cu_train = new_data[('Cu', 'train')]
material_rows = {}
for index, value in enumerate(cu_train['ids']):
    material_rows.setdefault(value.rsplit('_', 1)[0], []).append(index)
cu_materials = np.asarray(sorted(material_rows), dtype=object)
np.random.default_rng(SEED).shuffle(cu_materials)
target_rows = len(cu_train['ids']) / 2
selected_materials, selected_rows = [], []
for material in cu_materials:
    if len(selected_rows) >= target_rows:
        break
    selected_materials.append(str(material))
    selected_rows.extend(material_rows[str(material)])
if len(selected_rows) > target_rows and len(selected_materials) > 1:
    last = selected_materials[-1]
    without_last = len(selected_rows) - len(material_rows[last])
    if abs(without_last - target_rows) < abs(len(selected_rows) - target_rows):
        selected_materials.pop()
        selected_rows = [index for material in selected_materials for index in material_rows[material]]
selected_rows = sorted(selected_rows)
if not 0 < len(selected_rows) < len(cu_train['ids']):
    raise ValueError(f'Invalid half-Cu selection: {len(selected_rows)} of {len(cu_train["ids"])} rows')

DATASET_DIR.mkdir(parents=True, exist_ok=True)
HALF_SPLIT_PATH.write_text(''.join(cu_train['ids'][i] + '\n' for i in selected_rows), encoding='utf-8')
HALF_TEMP = tempfile.TemporaryDirectory(prefix='omnixas_half_cu_')
HALF_DATA_ROOT = Path(HALF_TEMP.name)
for folder in ('material_id_and_site', 'ml_data'):
    destination = HALF_DATA_ROOT / folder
    destination.mkdir(parents=True, exist_ok=False)
    for element in tqdm(ELEMENTS, desc='Build half-Cu dataset'):
        for split in SPLITS:
            suffix = '' if folder == 'material_id_and_site' else '_y'
            source = NEW_ROOT / folder / f'{element}_FEFF_{split}{suffix}.txt'
            target = destination / source.name
            if element == 'Cu' and split == 'train':
                if folder == 'material_id_and_site':
                    target.write_text(''.join(cu_train['ids'][i] + '\n' for i in selected_rows), encoding='utf-8')
                else:
                    np.savetxt(target, cu_train['y'][selected_rows], fmt='%.9g')
            else:
                shutil.copy2(source, target)

half_train_targets = cu_train['y'][selected_rows].copy()
print(f'Half-Cu subset: {len(selected_rows)}/{len(cu_train["ids"])} spectra from {len(selected_materials)}/{len(material_rows)} materials')
print(f'Saved half-Cu split: {HALF_SPLIT_PATH}')
print(f'Temporary training dataset: {HALF_DATA_ROOT}')
print(f'Run directory: {RUN_DIR}')

## Train the half-Cu pipeline

Runs the scratch M3GNet encoder, feature export, UniversalXAS training, and tuned-head training. Each pipeline's tqdm progress bars stream directly into this cell output.

In [ ]:
import runpy

training_args = [
    str(SCRIPT),
    '--dataset-root', str(HALF_DATA_ROOT),
    '--raw-root', str(RAW_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--run-name', RUN_NAME,
    '--seed', str(SEED),
    '--precision', 'bf16-mixed',
    '--encoder-rows-per-element', '64',
    '--encoder-eval-batch-size', '1024',
    '--num-workers', '32',
    '--batch-size', '4096',
    '--head-dropout', str(HEAD_DROPOUT),
    '--universal-loss-balance-power', str(UNIVERSAL_LOSS_BALANCE_POWER),
    '--universal-lr', str(UNIVERSAL_LEARNING_RATE),
    '--universal-seed', str(UNIVERSAL_SEED),
    '--universal-es-metric', 'macro_relative_median',
    '--tuned-lr', str(TUNED_LEARNING_RATE),
    '--tuned-seed', str(TUNED_SEED),
    '--tuned-schedule', 'plateau',
    '--tuned-loss-normalize',
]

previous_argv = sys.argv
sys.argv = training_args
try:
    runpy.run_path(str(SCRIPT), run_name='__main__')
finally:
    sys.argv = previous_argv

if not (RUN_DIR / 'RUN_COMPLETE.json').is_file():
    raise FileNotFoundError(f'Training did not complete: missing {RUN_DIR / "RUN_COMPLETE.json"}')
print(f'Half-Cu training complete: {RUN_DIR}')

## Evaluate Cu validation and test splits

Uses the same untouched Cu validation and test targets. Eta uses this run's own Cu training mean as the train-mean baseline.

In [ ]:
from omnixas.model.m3gnet_xas import XASSpectralHead

CU = 'Cu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cu_eval = {split: new_data[(CU, split)] for split in ('val', 'test')}

def load_features(run_dir, split):
    path = run_dir / 'features' / f'Cu_FEFF_{split}_X.txt'
    values = np.loadtxt(path, dtype=np.float32, ndmin=2)
    expected = len(cu_eval[split]['ids'])
    if values.shape != (expected, 64) or not np.isfinite(values).all():
        raise ValueError(f'Invalid Cu features for {run_dir.name} {split}: {values.shape}')
    return values

def load_head(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing checkpoint: {path}')
    head = XASSpectralHead(dropout=HEAD_DROPOUT).to(device)
    state = torch.load(path, map_location='cpu', weights_only=False)
    head.load_state_dict(state['state_dict'], strict=True)
    return head.eval()

def predict(head, features):
    batches = []
    with torch.inference_mode():
        for start in range(0, len(features), 4096):
            batch = torch.as_tensor(features[start:start + 4096], dtype=torch.float32, device=device)
            batches.append(head(batch).cpu().numpy())
    return np.concatenate(batches, axis=0)

def score(variant, split, targets, predictions, train_targets):
    errors = np.mean((targets - predictions) ** 2, axis=1)
    baseline_errors = np.mean((targets - train_targets.mean(axis=0, keepdims=True)) ** 2, axis=1)
    median_error = float(np.median(errors))
    return {
        'element': CU, 'training_regime': 'Half Cu train', 'variant': variant,
        'evaluation_split': split, 'n_spectra': len(targets),
        'aggregate_mse': float(np.mean(errors)),
        'median_spectrum_mse': median_error,
        'baseline_median_mse': float(np.median(baseline_errors)),
        'eta': float(np.median(baseline_errors) / max(median_error, 1e-12)),
    }, errors

records, test_predictions = [], {}
features = {split: load_features(RUN_DIR, split) for split in ('val', 'test')}
train_targets = half_train_targets
heads = {
    'UniversalXAS': RUN_DIR / 'heads' / 'universalXAS' / 'best.pt',
    'Tuned-UniversalXAS': RUN_DIR / 'heads' / 'tunedUniversalXAS' / 'Cu_FEFF' / 'best.pt',
}
for variant in tqdm(heads, desc='Heads'):
    head = load_head(heads[variant])
    for split in ('val', 'test'):
        targets = cu_eval[split]['y']
        predictions = predict(head, features[split])
        row, errors = score(variant, split, targets, predictions, train_targets)
        records.append(row)
        if split == 'test':
            test_predictions[(variant,)] = {'y': targets, 'pred': predictions, 'errors': errors}

results = pd.DataFrame(records)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(ANALYSIS_DIR / 'halfcu_cu_evaluation.csv', index=False)
display(results.round({'aggregate_mse': 6, 'median_spectrum_mse': 6, 'baseline_median_mse': 6, 'eta': 3}))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

percentiles = [10, 50, 90, 99]
energy = np.arange(141) * 0.25
half = test_predictions[('Tuned-UniversalXAS',)]
order = np.argsort(half['errors'])
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, percentile in zip(axes.flat, percentiles):
    index = order[int(round((percentile / 100) * (len(order) - 1)))]
    ax.plot(energy, half['y'][index], label='target', linewidth=1.6, color='black')
    ax.plot(energy, half['pred'][index], label='half-Cu train', linewidth=1.2, alpha=0.8)
    material_site = new_data[('Cu', 'test')]['ids'][index]
    material_id, site = material_site.rsplit('_', 1)
    ax.set_title(f'{percentile}th half-Cu error (MSE={half["errors"][index]:.3g})\n{material_id}-{int(site)}')
    ax.grid(alpha=0.2)
axes.flat[0].legend(frameon=False)
fig.supxlabel('Energy above grid start (eV)')
fig.supylabel('Unscaled intensity')
fig.suptitle('Cu unscaled Figshare test spectra: half-Cu training data')
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / 'Cu_unscaled_test_halfcu_percentiles.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Results saved in {ANALYSIS_DIR}')

In [ ]:
# ## Evaluate all elements on the test split
# Reports UniversalXAS and Tuned-UniversalXAS performance for all eight elements.
# Eta uses each element's own training mean as the train-mean baseline.

from omnixas.model.m3gnet_xas import XASSpectralHead

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_all_features(run_dir):
    features = {}
    for element in ELEMENTS:
        for split in ('val', 'test'):
            path = run_dir / 'features' / f'{element}_FEFF_{split}_X.txt'
            values = np.loadtxt(path, dtype=np.float32, ndmin=2)
            expected = len(new_data[(element, split)]['ids'])
            if values.shape != (expected, 64) or not np.isfinite(values).all():
                raise ValueError(f'Invalid features for {element} {split}: {values.shape}')
            features[(element, split)] = values
    return features

def load_head(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing checkpoint: {path}')
    head = XASSpectralHead(dropout=HEAD_DROPOUT).to(device)
    state = torch.load(path, map_location='cpu', weights_only=False)
    head.load_state_dict(state['state_dict'], strict=True)
    return head.eval()

def predict(head, features):
    batches = []
    with torch.inference_mode():
        for start in range(0, len(features), 4096):
            batch = torch.as_tensor(features[start:start + 4096], dtype=torch.float32, device=device)
            batches.append(head(batch).cpu().numpy())
    return np.concatenate(batches, axis=0)

def score_all(variant, element, split, targets, predictions, train_targets):
    errors = np.mean((targets - predictions) ** 2, axis=1)
    baseline_errors = np.mean((targets - train_targets.mean(axis=0, keepdims=True)) ** 2, axis=1)
    median_error = float(np.median(errors))
    return {
        'element': element, 'training_regime': 'Half Cu train', 'variant': variant,
        'evaluation_split': split, 'n_spectra': len(targets),
        'eta': float(np.median(baseline_errors) / max(median_error, 1e-12)),
    }

all_features = load_all_features(RUN_DIR)
train_targets = {element: new_data[(element, 'train')]['y'] for element in ELEMENTS}

universal_head = load_head(RUN_DIR / 'heads' / 'universalXAS' / 'best.pt')

records = []
for element in tqdm(ELEMENTS, desc='All elements'):
    tuned_head = load_head(RUN_DIR / 'heads' / 'tunedUniversalXAS' / f'{element}_FEFF' / 'best.pt')
    for variant, head in (('UniversalXAS', universal_head), ('Tuned-UniversalXAS', tuned_head)):
        for split in ('test',):
            targets = new_data[(element, split)]['y']
            predictions = predict(head, all_features[(element, split)])
            records.append(score_all(variant, element, split, targets, predictions, train_targets[element]))

all_results = pd.DataFrame(records)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
all_results.to_csv(ANALYSIS_DIR / 'halfcu_all_elements_evaluation.csv', index=False)
display(all_results.round({'eta': 3}))
